# AI Agent Demo — Natural-Language Explanation Generation

Generates human-readable explanations from model predictions and XAI evidence
using OpenAI API.

| Component | Description |
|---|---|
| Input | Model predictions + XAI artifacts (Phase 2-6) |
| Processing | Evidence compression → OpenAI prompt → Structured JSON |
| Output | Vietnamese/English reports (JSON + Markdown) |
| Branch | `xai-v3` |

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone and install

In [ ]:
!rm -rf /content/SE365
!git clone -b xai-v3 https://github.com/lechihoang/SE365.git /content/SE365
%cd /content/SE365
!pip install -q -r requirements.txt
!pip install -q openai>=1.0 jsonschema>=4.0 python-dotenv>=1.0

### STEP 3: Configuration

In [ ]:
import os, sys, json, time

DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_DIR      = f'{EXP_DIR}/xai'
AGENT_OUT    = f'{EXP_DIR}/agent_outputs'

os.makedirs(AGENT_OUT, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'EXP_DIR   : {EXP_DIR}')
print(f'XAI_DIR   : {XAI_DIR}')
print(f'AGENT_OUT : {AGENT_OUT}')

### STEP 4: Set OpenAI API Key

Load securely from Colab secrets or environment. **Never paste your key in code.**

In [ ]:
# Option A: Colab secrets (recommended)
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('API key loaded from Colab secrets.')
except Exception:
    print('Colab secrets not available. Ensure OPENAI_API_KEY is set.')

# Verify key is set (do NOT print it)
assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY not set!'
print(f'Key starts with: {os.environ["OPENAI_API_KEY"][:8]}...')

### STEP 5: Initialize Agent

In [ ]:
from agent import ExplanationAgent, AgentConfig

config = AgentConfig(
    batch_model='gpt-4o',
    report_model='gpt-4o',
    temperature=0.3,
    language='vi',
)

agent = ExplanationAgent(config)
print(f'Agent initialized.')
print(f'Batch model  : {config.batch_model}')
print(f'Report model : {config.report_model}')

### STEP 6: Load a Case Study Sample

In [ ]:
# Load the first case study from Phase 6 manifest
manifest_path = os.path.join(XAI_DIR, 'case_studies', 'sample_manifest.json')
if os.path.isfile(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f'Loaded manifest: {len(manifest)} case studies')
    # Show first entry
    if manifest:
        first = manifest[0]
        print(f'\nFirst case: {first["case_id"]}')
        print(f'  Sample: {first["sample_id"]}')
        print(f'  Type: {first["case_type"]}')
        print(f'  Reason: {first["reason"]}')
else:
    print(f'Manifest not found at {manifest_path}')
    manifest = []

### STEP 7: Preview Evidence Before API Call

In [ ]:
from agent.evidence_loader import EvidenceLoader
from agent.evidence_builder import EvidenceBuilder

if manifest:
    case = manifest[0]
    case_id = case['case_id']
    sample_id = case['sample_id']
    case_dir = os.path.join(XAI_DIR, 'case_studies', case_id)
    
    # Load case metadata
    meta_path = os.path.join(case_dir, 'metadata.json')
    with open(meta_path) as f:
        meta = json.load(f)
    
    # Load and compress evidence
    loader = EvidenceLoader()
    evidence = loader.load(sample_id, XAI_DIR, case_id)
    
    builder = EvidenceBuilder(config)
    blocks = builder.build(evidence)
    
    print('=== COMPRESSED EVIDENCE PREVIEW ===')
    for key, text in blocks.items():
        print(f'\n--- {key} ---')
        print(text[:300])
    
    print(f'\nMissing: {evidence.get("_missing", [])}')
else:
    print('No manifest loaded — skipping evidence preview.')

### STEP 8: Single Sample Explanation (API Call)

In [ ]:
if manifest:
    t0 = time.time()
    result = agent.explain_case_study(
        case_id=case_id,
        case_dir=case_dir,
        xai_dir=XAI_DIR,
        language='vi',
        output_dir=AGENT_OUT,
    )
    elapsed = time.time() - t0
    
    print(f'Generated in {elapsed:.1f}s')
    print(f'Confidence: {result.get("confidence", "?")}')
    
    # Display XAI visual artifacts alongside the explanation
    from PIL import Image as PILImage
    import matplotlib.pyplot as plt
    
    visuals = result.get('visual_artifacts', {})
    display_order = [
        ('gradcam_food', 'Grad-CAM (Food)'),
        ('attention_word_bar', 'Attention Word Importance'),
        ('cross_attention_overlay', 'Cross-Attention Token→Patch'),
        ('shap_chart', 'SHAP Modality Contribution'),
    ]
    shown = [(k, label) for k, label in display_order if visuals.get(k)]
    if shown:
        n = len(shown)
        fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
        if n == 1: axes = [axes]
        for ax, (key, label) in zip(axes, shown):
            img = PILImage.open(visuals[key])
            ax.imshow(img); ax.axis('off')
            ax.set_title(label, fontsize=10)
        plt.suptitle(f'XAI Visual Evidence — {sample_id}', fontsize=12)
        plt.tight_layout(); plt.show()
    
    # Show customer view
    cv = result.get('customer_view', {})
    if cv.get('summary'):
        print(f'\n=== CUSTOMER VIEW ===')
        print(cv['summary'])
        for h in cv.get('highlights', []):
            print(f'  • {h}')
    
    # Show technical summary
    print(f'\n=== TECHNICAL SUMMARY ===')
    print(result.get('summary', '(no summary)'))
    
    # Show evidence completeness
    ec = result.get('evidence_completeness', {})
    if ec:
        methods = ['gradcam', 'attention', 'cross_attention', 'shap', 'lime']
        status = ' | '.join(f'{m}={"✓" if ec.get(m) else "✗"}' for m in methods)
        print(f'\nEvidence: {status} ({ec.get("total", "?")})')
    
    warnings = result.get('validation_warnings', [])
    if warnings:
        print(f'\n=== WARNINGS ({len(warnings)}) ===')
        for w in warnings:
            print(f'  - {w}')
    else:
        print(f'\nValidation: PASSED (0 warnings)')
    
    print(f'\nSaved to: {AGENT_OUT}')
else:
    print('Skipping — no manifest.')

### STEP 9: Batch Mode — Process All Case Studies

In [ ]:
if manifest and len(manifest) > 1:
    # Build batch input from case study metadata
    batch_samples = []
    _d2t = {
        'Food Score': 'food_score',
        'Price Score': 'price_score',
        'Atmosphere Score': 'atmosphere_score',
        'Service Score': 'service_score',
        'Overall Satisfaction': 'overall_satisfaction',
    }
    
    for entry in manifest[:5]:  # Process first 5 for demo
        cid = entry['case_id']
        cd = os.path.join(XAI_DIR, 'case_studies', cid)
        mp = os.path.join(cd, 'metadata.json')
        if not os.path.isfile(mp):
            continue
        with open(mp) as f:
            m = json.load(f)
        preds = {_d2t.get(k, k): v for k, v in m.get('predictions', {}).items()}
        batch_samples.append({
            'sample_id': m.get('sample_id', cid),
            'review_text': m.get('review_text', ''),
            'predictions': preds,
            'case_type': m.get('case_type'),
        })
    
    print(f'Batch processing {len(batch_samples)} samples...')
    batch_results = agent.explain_batch(
        samples=batch_samples,
        xai_dir=XAI_DIR,
        language='vi',
        output_dir=AGENT_OUT,
    )
    
    n_ok = sum(1 for r in batch_results if 'error' not in r)
    print(f'\nBatch complete: {n_ok}/{len(batch_results)} succeeded')
    print(f'Outputs saved to: {AGENT_OUT}')
else:
    print('Not enough samples for batch demo.')

### STEP 10: Final Summary

In [ ]:
print('='*60)
print('  AI AGENT DEMO — SUMMARY')
print('='*60)

if os.path.isdir(AGENT_OUT):
    files = os.listdir(AGENT_OUT)
    print(f'  Output dir : {AGENT_OUT}')
    print(f'  Files      : {len(files)}')
    for f in sorted(files):
        print(f'    {f}')
else:
    print('  No outputs generated.')

print()
print('  The AI Agent converts XAI evidence into human-readable')
print('  explanations via OpenAI API. All claims are grounded')
print('  in existing Grad-CAM, Attention, SHAP, LIME, and')
print('  Cross-Attention artifacts.')
print('='*60)